# GS Tax (Grunnrenteskatt) Price Adjustment

Norwegian hydro generators are subject to a **resource rent tax** (grunnrenteskatt, commonly called "GS tax"). The Nordic regulator requires that BSPs should not make more money in mFRR than is possible in the day-ahead auction.

This means bid prices are **not static per tier** – they vary per MTU because the day-ahead price varies per MTU.

## Formula

```
bid_price = tier_price + tax_rate × (DA_price − tier_price)
```

- `tier_price` – base price the trader has set for this tier (EUR/MWh)
- `DA_price` – day-ahead market price for this MTU (EUR/MWh)
- `tax_rate` – resource rent tax rate as a decimal (e.g. `0.59` for 59%)

After applying the formula, the result is clamped:

- **Up bids**: bid price must be `≥ DA_price`
- **Down bids**: bid price must be `≤ DA_price`
- **Statnett limits**: ±9,999 EUR/MWh (pre-MARI) or ±14,999 EUR/MWh (post-MARI)
- Rounded to 2 decimal places (0.01 EUR granularity)

## 1. Setup

In [1]:
from datetime import UTC, datetime, timedelta
from decimal import Decimal

import pandas as pd

from nexa_mfrr_eam import (
    Bid,
    BiddingZone,
    Direction,
    MarketProductType,
    gs_adjust_bids,
    gs_adjusted_price,
)

# Norwegian hydro generator
TAX_RATE = Decimal("0.59")  # 59% resource rent tax rate
RESOURCE_ID = "NOKG90901"
BIDDING_ZONE = BiddingZone.NO2

# Three upward price tiers (EUR/MWh)
TIER_PRICES = [Decimal("145.00"), Decimal("185.00"), Decimal("225.00")]

# Base MTU for the example
BASE_MTU = datetime(2026, 3, 21, 8, 0, tzinfo=UTC)

## 2. Day-Ahead Prices

We use a simple dict of DA prices for 12 consecutive MTUs. In production these would come from a market data source.

In [2]:
# Realistic NO2 DA prices (EUR/MWh) for 12 MTUs
raw_da_prices = [
    95.30, 98.10, 102.40, 108.75, 115.20, 122.80,
    131.73, 138.50, 145.00, 152.10, 148.60, 141.90,
]

da_prices: dict[datetime, Decimal] = {
    BASE_MTU + timedelta(minutes=15 * i): Decimal(str(p))
    for i, p in enumerate(raw_da_prices)
}

pd.Series(
    {k.strftime("%H:%M"): float(v) for k, v in da_prices.items()},
    name="DA price (EUR/MWh)",
).to_frame()

,DA price (EUR/MWh)
08:00,95.30
08:15,98.10
08:30,102.40
08:45,108.75
09:00,115.20
09:15,122.80
09:30,131.73
09:45,138.50
10:00,145.00
10:15,152.10


## 3. Single Price Calculation (Step-by-Step)

Let's trace through the formula manually for one MTU to understand what the function does.

In [3]:
tier_price = Decimal("185.00")
da_price = Decimal("131.73")
tax_rate = TAX_RATE

raw_adjusted = tier_price + tax_rate * (da_price - tier_price)
print(f"  tier_price  = {tier_price}")
print(f"  DA_price    = {da_price}")
print(f"  tax_rate    = {tax_rate}")
print()
print(f"  adjusted = {tier_price} + {tax_rate} × ({da_price} − {tier_price})")
print(f"           = {tier_price} + {tax_rate} × ({da_price - tier_price})")
print(f"           = {raw_adjusted:.2f}")
print()
print(f"  UP clamp: max({raw_adjusted:.2f}, {da_price}) = {max(raw_adjusted, da_price):.2f}")
print()

# Using the library function
result = gs_adjusted_price(
    tier_price=tier_price,
    da_price=da_price,
    tax_rate=tax_rate,
    direction=Direction.UP,
)
print(f"gs_adjusted_price(...) = {result}")

  tier_price  = 185.00
  DA_price    = 131.73
  tax_rate    = 0.59

  adjusted = 185.00 + 0.59 × (131.73 − 185.00)
           = 185.00 + 0.59 × (-53.27)
           = 153.57

  UP clamp: max(153.57, 131.73) = 153.57

gs_adjusted_price(...) = 153.57


## 4. Batch Adjustment: 12 MTUs × 3 Tiers = 36 Bids

Build a set of bids at static tier prices and then adjust them for GS tax using `gs_adjust_bids`.

In [4]:
# Build static bids (one per MTU per tier)
static_bids = [
    Bid.up(volume_mw=20, price_eur=float(tier))
    .divisible(min_volume_mw=10)
    .for_mtu(mtu)
    .bidding_zone(BIDDING_ZONE)
    .resource(RESOURCE_ID, coding_scheme="NNO")
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .build()
    for mtu in da_prices
    for tier in TIER_PRICES
]

print(f"Built {len(static_bids)} bids")

# Apply GS adjustment
adjusted_bids = gs_adjust_bids(
    bids=static_bids,
    da_prices=da_prices,
    tax_rate=TAX_RATE,
)

print(f"Adjusted {len(adjusted_bids)} bids")

Built 36 bids
Adjusted 36 bids


In [5]:
# Build a before/after comparison table
rows = []
for orig, adj in zip(static_bids, adjusted_bids):
    mtu = orig.period.time_interval_start
    da = da_prices[mtu]
    tier = orig.period.point.energy_price
    adjusted = adj.period.point.energy_price
    rows.append({
        "MTU": mtu.strftime("%H:%M"),
        "DA price": float(da),
        "Tier price": float(tier),
        "Adjusted price": float(adjusted),
        "Change": float(adjusted - tier),
    })

df = pd.DataFrame(rows)
df.style.format({
    "DA price": "{:.2f}",
    "Tier price": "{:.2f}",
    "Adjusted price": "{:.2f}",
    "Change": "{:+.2f}",
})

,MTU,DA price,Tier price,Adjusted price,Change
0,08:00,95.30,145.00,115.68,-29.32
1,08:00,95.30,185.00,132.08,-52.92
2,08:00,95.30,225.00,148.48,-76.52
3,08:15,98.10,145.00,117.33,-27.67
4,08:15,98.10,185.00,133.73,-51.27
5,08:15,98.10,225.00,150.13,-74.87
6,08:30,102.40,145.00,119.87,-25.13
7,08:30,102.40,185.00,136.27,-48.73
8,08:30,102.40,225.00,152.67,-72.33
9,08:45,108.75,145.00,123.61,-21.39


## 5. Clamping in Action

When the DA price is high, the formula can produce an adjusted price below the DA price for an UP bid. The clamp ensures the bid price is never below DA.

In this example, a tier price of **145 EUR/MWh** with a DA price of **145 EUR/MWh** (or higher) demonstrates the clamp.

In [6]:
clamp_examples = []
for da in [120.0, 130.0, 145.0, 160.0, 180.0]:
    for tier in [145.0]:
        raw = Decimal(str(tier)) + TAX_RATE * (Decimal(str(da)) - Decimal(str(tier)))
        clamped = gs_adjusted_price(tier, da, TAX_RATE, Direction.UP)
        clamp_examples.append({
            "Tier price": tier,
            "DA price": da,
            "Formula result": float(raw.quantize(Decimal("0.01"))),
            "Clamped result": float(clamped),
            "Clamp applied?": clamped > raw.quantize(Decimal("0.01")),
        })

pd.DataFrame(clamp_examples).style.format(
    {"Tier price": "{:.2f}", "DA price": "{:.2f}",
     "Formula result": "{:.2f}", "Clamped result": "{:.2f}"}
)

,Tier price,DA price,Formula result,Clamped result,Clamp applied?
0,145.00,120.00,130.25,130.25,False
1,145.00,130.00,136.15,136.15,False
2,145.00,145.00,145.00,145.00,False
3,145.00,160.00,153.85,160.00,True
4,145.00,180.00,165.65,180.00,True


When the DA price is ≥ the tier price (e.g. 145 or 160 EUR/MWh against a 145 EUR tier), the formula produces a result equal to or below the DA price. The UP clamp kicks in and sets the bid price to the DA price.

## 6. Down Bid Example

For down bids the clamping rule is reversed: the adjusted price must be **≤ DA price**.

Consider two down bid tiers: 40 EUR/MWh and 20 EUR/MWh.

In [7]:
down_examples = []
for da in [60.0, 80.0, 100.0, 120.0]:
    for tier in [40.0, 20.0]:
        raw = Decimal(str(tier)) + TAX_RATE * (Decimal(str(da)) - Decimal(str(tier)))
        clamped = gs_adjusted_price(tier, da, TAX_RATE, Direction.DOWN)
        down_examples.append({
            "Tier price": tier,
            "DA price": da,
            "Formula result": float(raw.quantize(Decimal("0.01"))),
            "Clamped result": float(clamped),
            "Clamp applied?": clamped < raw.quantize(Decimal("0.01")),
        })

pd.DataFrame(down_examples).style.format(
    {"Tier price": "{:.2f}", "DA price": "{:.2f}",
     "Formula result": "{:.2f}", "Clamped result": "{:.2f}"}
)

,Tier price,DA price,Formula result,Clamped result,Clamp applied?
0,40.00,60.00,51.80,51.80,False
1,20.00,60.00,43.60,43.60,False
2,40.00,80.00,63.60,63.60,False
3,20.00,80.00,55.40,55.40,False
4,40.00,100.00,75.40,75.40,False
5,20.00,100.00,67.20,67.20,False
6,40.00,120.00,87.20,87.20,False
7,20.00,120.00,79.00,79.00,False


For DOWN bids with a low tier price and high DA price, the formula produces a value higher than the DA price. The DOWN clamp sets it back to the DA price.

## Summary

- `gs_adjusted_price` applies the formula to a single tier/MTU and returns a clamped, rounded `Decimal`.
- `gs_adjust_bids` applies the adjustment to a whole list of `BidTimeSeriesModel` objects, returning new (non-mutated) instances.
- Use `mariMode` to switch between pre-MARI (±9,999) and post-MARI (±14,999) Statnett limits.